# ShovelSense Data Exploration
### Automated Smart Truck Diversion — EDA & ML Candidate Analysis

This notebook explores the **fact** and **dimension** tables in `cjc_aws_workspace_catalog.shovelsense` to:
1. Understand data structure, volumes, and quality
2. Visualize key distributions, trends, and relationships
3. Identify strong candidates for **regression** and **classification** models

**Domain context**: ShovelSense uses XRF sensors on mining shovels to classify material (ore vs. waste) in real-time, enabling smart truck diversion decisions that recover misclassified ore and prevent dilution.

---

**Table of Contents**
- [1. Setup & Configuration](#section-setup)
- [2. Schema Overview](#section-schema)
- [3. Dimension Tables](#section-dimensions)
- [4. Fact Tables — Truck Loads & Bucket Measurements](#section-facts-core)
- [5. Fact Tables — Classification & Diversions](#section-facts-class)
- [6. Key Visualizations](#section-viz)
- [7. ML Candidate Analysis](#section-ml)

In [0]:
# Setup & Configuration
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (14, 5)})

CATALOG = "cjc_aws_workspace_catalog"
SCHEMA = "shovelsense"
DB = f"{CATALOG}.{SCHEMA}"

print(f"Target schema: {DB}")

## <a id="section-schema"></a>2. Schema Overview
Row counts and column counts for every table in the schema.

In [0]:
# Collect metadata for every table
tables = spark.sql(f"SHOW TABLES IN {DB}").toPandas()

rows_list = []
for _, row in tables.iterrows():
    tbl = f"{DB}.{row['tableName']}"
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {tbl}").collect()[0]["cnt"]
    cols = len(spark.table(tbl).columns)
    tbl_type = "dim" if row['tableName'].startswith('dim_') else (
        "fact" if row['tableName'].startswith('fact_') else (
        "summary" if row['tableName'].startswith('summary_') else (
        "bronze" if row['tableName'].startswith('bronze_') else "silver")))
    rows_list.append({"table": row['tableName'], "layer": tbl_type, "row_count": cnt, "col_count": cols})

schema_df = pd.DataFrame(rows_list).sort_values(["layer", "table"])
display(schema_df)

In [0]:
# Visualize table sizes
fig = px.bar(
    schema_df,
    x="table", y="row_count",
    color="layer",
    title="Row Counts by Table",
    labels={"row_count": "Rows", "table": ""},
    color_discrete_map={"dim": "#1f77b4", "fact": "#ff7f0e", "bronze": "#8c564b", "silver": "#7f7f7f", "summary": "#2ca02c"}
)
fig.update_layout(xaxis_tickangle=-45, height=450)
fig.show()

## <a id="section-dimensions"></a>3. Dimension Tables
Profile the four dimension tables: **trucks**, **shovels**, **block model**, and **date**.

In [0]:
# -- dim_trucks --
df_trucks = spark.table(f"{DB}.dim_trucks").toPandas()
print(f"dim_trucks: {len(df_trucks)} rows, {df_trucks.shape[1]} columns")
print(f"\nTruck models: {df_trucks['truck_model'].unique()}")
print(f"Payload capacities: {sorted(df_trucks['payload_capacity_tonnes'].unique())}")
print(f"FMS systems: {df_trucks['fms_system'].unique()}")
display(df_trucks)

In [0]:
# -- dim_shovels --
df_shovels = spark.table(f"{DB}.dim_shovels").toPandas()
print(f"dim_shovels: {len(df_shovels)} rows, {df_shovels.shape[1]} columns")
print(f"\nShovel types: {df_shovels['shovel_type'].unique()}")
print(f"MineSense equipped: {df_shovels['minesense_equipped'].value_counts().to_dict()}")
print(f"Sensor heads: {sorted(df_shovels['sensor_heads'].dropna().unique())}")
display(df_shovels)

In [0]:
# -- dim_block_model --
df_blocks = spark.table(f"{DB}.dim_block_model").toPandas()
print(f"dim_block_model: {len(df_blocks)} rows, {df_blocks.shape[1]} columns")
print(f"\nPlanned classifications: {df_blocks['planned_classification'].value_counts().to_dict()}")
print(f"Geological domains: {df_blocks['geological_domain'].value_counts().to_dict()}")
print(f"Mineralogy classes: {df_blocks['mineralogy_class'].value_counts().to_dict()}")
print(f"\nCu grade range: {df_blocks['planned_cu_grade'].min():.4f} - {df_blocks['planned_cu_grade'].max():.4f}")
print(f"Fe grade range: {df_blocks['planned_fe_grade'].min():.4f} - {df_blocks['planned_fe_grade'].max():.4f}")

In [0]:
# Block model spatial visualization
fig = px.scatter(
    df_blocks,
    x="easting", y="northing",
    color="planned_classification",
    size="planned_cu_grade",
    hover_data=["bench", "geological_domain", "planned_cu_grade", "grade_bin"],
    title="Block Model — Spatial Distribution by Classification",
    labels={"easting": "Easting (m)", "northing": "Northing (m)"},
    opacity=0.6
)
fig.update_layout(height=550)
fig.show()

In [0]:
# Block model — Cu grade distribution by geological domain (matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Cu grade histogram by classification
for cls in df_blocks['planned_classification'].unique():
    subset = df_blocks[df_blocks['planned_classification'] == cls]
    axes[0].hist(subset['planned_cu_grade'], bins=40, alpha=0.6, label=cls)
axes[0].set_xlabel('Planned Cu Grade (%)')
axes[0].set_ylabel('Block Count')
axes[0].set_title('Cu Grade Distribution by Planned Classification')
axes[0].legend()

# Mineralogy pie chart
min_counts = df_blocks['mineralogy_class'].value_counts()
axes[1].pie(min_counts, labels=min_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Mineralogy Class Distribution')

plt.tight_layout()
plt.show()

## <a id="section-facts-core"></a>4. Fact Tables — Truck Loads & Bucket Measurements
The two highest-volume fact tables containing individual load and bucket-level sensor readings.

In [0]:
# -- fact_truck_loads -- (main fact table)
df_loads = spark.table(f"{DB}.fact_truck_loads").drop("timestamp").toPandas()
print(f"fact_truck_loads: {len(df_loads):,} rows, {df_loads.shape[1]} columns")
print(f"\nDate range: {df_loads['load_date'].min()} to {df_loads['load_date'].max()}")
print(f"Unique trucks: {df_loads['truck_id'].nunique()}")
print(f"Unique shovels: {df_loads['shovel_id'].nunique()}")
print(f"Unique blocks: {df_loads['block_id'].nunique()}")
print(f"\nPlanned classifications: {df_loads['planned_classification'].value_counts().to_dict()}")
print(f"ShovelSense classifications: {df_loads['shovelsense_classification'].value_counts().to_dict()}")
print(f"Diversion types: {df_loads['diversion_type'].value_counts().to_dict()}")
print(f"\nDiversion rate: {df_loads['is_diverted'].mean():.2%}")
print(f"Ore recovery rate: {df_loads['is_ore_recovery'].mean():.2%}")
print(f"Dilution prevention rate: {df_loads['is_dilution_prevention'].mean():.2%}")
print(f"\nNumeric summary:")
display(df_loads.describe())

In [0]:
# Truck loads — classification comparison (from user's reference image)
class_comparison = df_loads.groupby(['planned_classification', 'shovelsense_classification', 'diversion_type']).size().reset_index(name='count')
print("Classification comparison (Planned vs ShovelSense):")
display(class_comparison)

# Confusion-matrix style heatmap
confusion = pd.crosstab(df_loads['planned_classification'], df_loads['shovelsense_classification'], margins=True)
print("\nConfusion matrix (Planned vs ShovelSense):")
display(confusion)

In [0]:
# Diversion type breakdown (Plotly sunburst)
diversion_data = df_loads.groupby(['planned_classification', 'shovelsense_classification', 'diversion_type']).agg(
    count=('load_id', 'size'),
    avg_cu=('avg_cu_grade_pct', 'mean'),
    total_value=('estimated_cu_value_usd', 'sum')
).reset_index()

fig = px.sunburst(
    diversion_data,
    path=['planned_classification', 'shovelsense_classification', 'diversion_type'],
    values='count',
    color='avg_cu',
    color_continuous_scale='YlOrRd',
    title='Truck Load Classification Flow (Planned → ShovelSense → Diversion)',
    labels={'avg_cu': 'Avg Cu Grade %'}
)
fig.update_layout(height=600)
fig.show()

In [0]:
# Load volume and grade trends over time
daily_loads = df_loads.groupby('load_date').agg(
    n_loads=('load_id', 'size'),
    avg_cu=('avg_cu_grade_pct', 'mean'),
    avg_fe=('avg_fe_grade_pct', 'mean'),
    diversion_rate=('is_diverted', 'mean'),
    avg_payload=('payload_tonnes', 'mean'),
    avg_xrf_conf=('avg_xrf_confidence', 'mean')
).reset_index()

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=('Daily Load Count', 'Avg Cu Grade & Diversion Rate', 'Avg XRF Confidence'),
    vertical_spacing=0.08
)
fig.add_trace(go.Bar(x=daily_loads['load_date'], y=daily_loads['n_loads'], name='Loads', marker_color='steelblue'), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_loads['load_date'], y=daily_loads['avg_cu'], name='Avg Cu %', line=dict(color='orange')), row=2, col=1)
fig.add_trace(go.Scatter(x=daily_loads['load_date'], y=daily_loads['diversion_rate'], name='Diversion Rate', line=dict(color='red', dash='dot'), yaxis='y2'), row=2, col=1)
fig.add_trace(go.Scatter(x=daily_loads['load_date'], y=daily_loads['avg_xrf_conf'], name='XRF Confidence', line=dict(color='green')), row=3, col=1)

fig.update_layout(height=700, title_text='Truck Loads — Daily Trends', showlegend=True)
fig.show()

In [0]:
# -- fact_bucket_measurements --
df_buckets = spark.table(f"{DB}.fact_bucket_measurements").drop("timestamp").toPandas()
print(f"fact_bucket_measurements: {len(df_buckets):,} rows, {df_buckets.shape[1]} columns")
print(f"\nDate range: {df_buckets['measurement_date'].min()} to {df_buckets['measurement_date'].max()}")
print(f"Grade categories: {df_buckets['grade_category'].value_counts().to_dict()}")
print(f"Matrix effect severities: {df_buckets['matrix_effect_severity'].value_counts().to_dict()}")
print(f"High confidence rate: {df_buckets['is_high_confidence'].mean():.2%}")
print(f"Both sensors active rate: {df_buckets['both_sensors_active'].mean():.2%}")
print(f"\nNumeric summary:")
display(df_buckets[['cu_grade_pct', 'fe_grade_pct', 'zn_grade_ppm', 'as_grade_ppm', 
                    'laser_fill_level_pct', 'xrf_confidence', 'matrix_effect_factor',
                    'measurement_quality_score']].describe())

In [0]:
# Bucket measurement distributions (matplotlib)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(df_buckets['cu_grade_pct'].dropna(), bins=50, color='darkorange', edgecolor='white')
axes[0, 0].set_title('Cu Grade Distribution')
axes[0, 0].set_xlabel('Cu Grade %')

axes[0, 1].hist(df_buckets['fe_grade_pct'].dropna(), bins=50, color='steelblue', edgecolor='white')
axes[0, 1].set_title('Fe Grade Distribution')
axes[0, 1].set_xlabel('Fe Grade %')

axes[0, 2].hist(df_buckets['xrf_confidence'].dropna(), bins=50, color='seagreen', edgecolor='white')
axes[0, 2].set_title('XRF Confidence Distribution')
axes[0, 2].set_xlabel('Confidence')

axes[1, 0].hist(df_buckets['laser_fill_level_pct'].dropna(), bins=50, color='purple', edgecolor='white')
axes[1, 0].set_title('Laser Fill Level Distribution')
axes[1, 0].set_xlabel('Fill Level %')

axes[1, 1].hist(df_buckets['measurement_quality_score'].dropna(), bins=50, color='crimson', edgecolor='white')
axes[1, 1].set_title('Measurement Quality Score')
axes[1, 1].set_xlabel('Quality Score')

for severity in df_buckets['matrix_effect_severity'].unique():
    subset = df_buckets[df_buckets['matrix_effect_severity'] == severity]
    axes[1, 2].hist(subset['matrix_effect_factor'].dropna(), bins=30, alpha=0.5, label=severity)
axes[1, 2].set_title('Matrix Effect Factor by Severity')
axes[1, 2].set_xlabel('Matrix Effect Factor')
axes[1, 2].legend(fontsize=8)

plt.suptitle('Bucket Measurement Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [0]:
# Bucket-level: Cu vs Fe colored by grade category (Plotly)
fig = px.scatter(
    df_buckets.sample(min(5000, len(df_buckets)), random_state=42),
    x='cu_grade_pct', y='fe_grade_pct',
    color='grade_category',
    size='xrf_confidence',
    hover_data=['measurement_quality_score', 'matrix_effect_severity'],
    title='Bucket Measurements — Cu vs Fe Grade (sampled)',
    labels={'cu_grade_pct': 'Cu Grade %', 'fe_grade_pct': 'Fe Grade %'},
    opacity=0.5
)
fig.update_layout(height=500)
fig.show()

## <a id="section-facts-class"></a>5. Fact Tables — Classification Accuracy & Diversions
Pre-aggregated tables for classification performance, daily diversions, sensor health, grade distributions, and surface-volume correlation.

In [0]:
# -- fact_classification_accuracy --
df_class_acc = spark.table(f"{DB}.fact_classification_accuracy").toPandas()
df_class_acc = df_class_acc.sort_values('load_date')
print(f"fact_classification_accuracy: {len(df_class_acc)} rows")
print(f"Accuracy range: {df_class_acc['accuracy'].min():.3f} - {df_class_acc['accuracy'].max():.3f}")
print(f"F1 score range: {df_class_acc['f1_score'].min():.3f} - {df_class_acc['f1_score'].max():.3f}")

# Classification accuracy over time (Plotly)
fig = go.Figure()
for metric, color in [('accuracy', 'blue'), ('precision_ore', 'green'), ('recall_ore', 'orange'), ('f1_score', 'red')]:
    fig.add_trace(go.Scatter(x=df_class_acc['load_date'], y=df_class_acc[metric], name=metric.replace('_', ' ').title(), line=dict(color=color)))

fig.update_layout(
    title='Classification Accuracy Metrics Over Time',
    xaxis_title='Date', yaxis_title='Score',
    height=450, yaxis=dict(range=[0, 1.05])
)
fig.show()

In [0]:
# -- fact_daily_diversions --
df_diversions = spark.table(f"{DB}.fact_daily_diversions").toPandas()
df_diversions = df_diversions.sort_values('load_date')
print(f"fact_daily_diversions: {len(df_diversions)} rows")
print(f"Total diverted trucks: {df_diversions['diverted_trucks'].sum():,}")
print(f"Total ore-from-waste: {df_diversions['ore_from_waste_count'].sum():,}")
print(f"Total waste-from-ore: {df_diversions['waste_from_ore_count'].sum():,}")

# Daily diversions by shovel (Plotly)
fig = px.bar(
    df_diversions,
    x='load_date', y='diverted_trucks',
    color='shovel_id',
    title='Daily Diverted Trucks by Shovel',
    labels={'diverted_trucks': 'Diverted Trucks', 'load_date': 'Date'},
    barmode='stack'
)
fig.update_layout(height=450)
fig.show()

In [0]:
# Diversion rate and recovery metrics over time (matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Aggregate by date across shovels
daily_agg = df_diversions.groupby('load_date').agg(
    total=('total_trucks', 'sum'),
    diverted=('diverted_trucks', 'sum'),
    ore_waste=('ore_from_waste_count', 'sum'),
    waste_ore=('waste_from_ore_count', 'sum')
).reset_index()
daily_agg['div_rate'] = daily_agg['diverted'] / daily_agg['total']

axes[0].plot(daily_agg['load_date'], daily_agg['div_rate'], color='crimson', linewidth=1.5)
axes[0].fill_between(daily_agg['load_date'], daily_agg['div_rate'], alpha=0.2, color='crimson')
axes[0].set_title('Overall Diversion Rate Over Time')
axes[0].set_ylabel('Diversion Rate')
axes[0].tick_params(axis='x', rotation=45)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

axes[1].stackplot(
    daily_agg['load_date'],
    daily_agg['ore_waste'], daily_agg['waste_ore'],
    labels=['Ore from Waste', 'Waste from Ore'],
    colors=['#2ca02c', '#d62728'], alpha=0.7
)
axes[1].set_title('Daily Diversion Types')
axes[1].set_ylabel('Count')
axes[1].legend(loc='upper right')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# -- fact_sensor_performance --
df_sensor = spark.table(f"{DB}.fact_sensor_performance").toPandas()
df_sensor = df_sensor.sort_values('measurement_date')
print(f"fact_sensor_performance: {len(df_sensor)} rows")
print(f"Avg confidence range: {df_sensor['avg_confidence'].min():.3f} - {df_sensor['avg_confidence'].max():.3f}")

fig = px.line(
    df_sensor,
    x='measurement_date', y='high_confidence_rate',
    color='shovel_id',
    title='Sensor High-Confidence Rate by Shovel Over Time',
    labels={'high_confidence_rate': 'High Confidence Rate', 'measurement_date': 'Date'}
)
fig.update_layout(height=400)
fig.show()

In [0]:
# -- fact_grade_distribution & fact_sv_correlation_analysis --
df_grade_dist = spark.table(f"{DB}.fact_grade_distribution").toPandas()
df_sv_corr = spark.table(f"{DB}.fact_sv_correlation_analysis").toPandas()

print(f"fact_grade_distribution: {len(df_grade_dist)} rows")
print(f"fact_sv_correlation_analysis: {len(df_sv_corr)} rows")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Grade distribution by domain
for domain in df_grade_dist['geological_domain'].unique():
    subset = df_grade_dist[df_grade_dist['geological_domain'] == domain]
    axes[0].barh(subset['grade_bin'], subset['block_count'], alpha=0.6, label=domain)
axes[0].set_xlabel('Block Count')
axes[0].set_title('Grade Distribution by Geological Domain')
axes[0].legend(fontsize=8)

# SV correlation vs accuracy
axes[1].bar(df_sv_corr['sv_corr_bin'], df_sv_corr['accuracy_rate'], color='steelblue', alpha=0.7, label='Accuracy')
ax2 = axes[1].twinx()
ax2.plot(df_sv_corr['sv_corr_bin'], df_sv_corr['diversion_rate'], color='red', marker='o', label='Diversion Rate')
axes[1].set_xlabel('SV Correlation Bin')
axes[1].set_ylabel('Accuracy Rate')
ax2.set_ylabel('Diversion Rate')
axes[1].set_title('SV Correlation: Accuracy vs Diversion Rate')
axes[1].legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [0]:
# -- fact_domain_classification_accuracy --
df_domain_acc = spark.table(f"{DB}.fact_domain_classification_accuracy").toPandas()
print(f"fact_domain_classification_accuracy: {len(df_domain_acc)} rows")

fig = px.bar(
    df_domain_acc.sort_values('accuracy', ascending=False),
    x='geological_domain',
    y=['accuracy', 'precision_ore', 'recall_ore', 'f1_score'],
    barmode='group',
    title='Classification Accuracy by Geological Domain',
    labels={'value': 'Score', 'geological_domain': 'Geological Domain'},
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(height=450, xaxis_tickangle=-30)
fig.show()

## <a id="section-viz"></a>6. Key Visualizations
Cross-cutting visualizations combining data from multiple tables.

In [0]:
# Payload vs Cu grade colored by diversion status (Plotly)
fig = px.scatter(
    df_loads.sample(min(5000, len(df_loads)), random_state=42),
    x='payload_tonnes', y='avg_cu_grade_pct',
    color='diversion_type',
    size='avg_xrf_confidence',
    facet_col='shift',
    title='Payload vs Cu Grade by Diversion Type & Shift',
    labels={'payload_tonnes': 'Payload (tonnes)', 'avg_cu_grade_pct': 'Avg Cu Grade %'},
    opacity=0.5,
    height=450
)
fig.show()

In [0]:
# Cycle time analysis by destination and diversion type (matplotlib)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Box plot: cycle time by diversion type
diversion_types = df_loads['diversion_type'].unique()
data_by_div = [df_loads[df_loads['diversion_type'] == dt]['cycle_time_minutes'].dropna() for dt in diversion_types]
bp1 = axes[0].boxplot(data_by_div, labels=diversion_types, patch_artist=True)
colors_box = plt.cm.Set2(np.linspace(0, 1, len(diversion_types)))
for patch, color in zip(bp1['boxes'], colors_box):
    patch.set_facecolor(color)
axes[0].set_ylabel('Cycle Time (min)')
axes[0].set_title('Cycle Time by Diversion Type')
axes[0].tick_params(axis='x', rotation=30)

# Box plot: Cu grade by destination
destinations = df_loads['destination'].unique()
data_by_dest = [df_loads[df_loads['destination'] == d]['avg_cu_grade_pct'].dropna() for d in destinations]
bp2 = axes[1].boxplot(data_by_dest, labels=destinations, patch_artist=True)
colors_dest = plt.cm.Set3(np.linspace(0, 1, len(destinations)))
for patch, color in zip(bp2['boxes'], colors_dest):
    patch.set_facecolor(color)
axes[1].set_ylabel('Avg Cu Grade %')
axes[1].set_title('Cu Grade by Destination')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [0]:
# Shift-level heatmap: avg Cu grade by hour and day of week (Plotly)
heatmap_data = df_loads.pivot_table(
    values='avg_cu_grade_pct', 
    index='load_hour', 
    columns='day_of_week', 
    aggfunc='mean'
)

fig = px.imshow(
    heatmap_data,
    labels=dict(x='Day of Week (1=Mon)', y='Load Hour', color='Avg Cu Grade %'),
    title='Avg Cu Grade by Hour & Day of Week',
    color_continuous_scale='YlOrRd',
    aspect='auto'
)
fig.update_layout(height=500)
fig.show()

## <a id="section-ml"></a>7. ML Candidate Analysis

We evaluate the suitability of key datasets for **regression** and **classification** models.

### Classification targets
| Target | Description | Source |
|--------|------------|--------|
| `is_diverted` | Whether the truck was diverted from planned route | fact_truck_loads |
| `shovelsense_classification` | Ore vs waste classification by sensor | fact_truck_loads |
| `diversion_type` | Multi-class: Aligned / Ore from Waste / Waste from Ore | fact_truck_loads |
| `grade_category` | Grade class from bucket measurements | fact_bucket_measurements |

### Regression targets
| Target | Description | Source |
|--------|------------|--------|
| `avg_cu_grade_pct` | Copper grade — continuous value | fact_truck_loads |
| `estimated_cu_value_usd` | Economic value of copper per load | fact_truck_loads |
| `measurement_quality_score` | Sensor quality metric | fact_bucket_measurements |

In [0]:
# Feature correlation heatmap for fact_truck_loads (matplotlib + seaborn)
numeric_cols = ['n_buckets', 'avg_cu_grade_pct', 'avg_fe_grade_pct', 'payload_tonnes',
                'cycle_time_minutes', 'estimated_cu_tonnes', 'avg_xrf_confidence',
                'surface_volume_correlation', 'estimated_cu_value_usd', 'load_hour', 'day_of_week']

corr_matrix = df_loads[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix — fact_truck_loads', fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
# Classification candidate: feature distributions by diversion_type (Plotly)
features_to_plot = ['avg_cu_grade_pct', 'avg_fe_grade_pct', 'avg_xrf_confidence',
                    'surface_volume_correlation', 'payload_tonnes', 'cycle_time_minutes']

fig = make_subplots(rows=2, cols=3, subplot_titles=features_to_plot)
for i, feat in enumerate(features_to_plot):
    row, col = (i // 3) + 1, (i % 3) + 1
    for div_type in df_loads['diversion_type'].dropna().unique():
        subset = df_loads[df_loads['diversion_type'] == div_type][feat].dropna()
        fig.add_trace(
            go.Histogram(x=subset, name=div_type, opacity=0.6, nbinsx=40,
                        showlegend=(i == 0)),
            row=row, col=col
        )

fig.update_layout(
    height=600, 
    title_text='Feature Distributions by Diversion Type (Classification Target)',
    barmode='overlay'
)
fig.show()

In [0]:
# Regression candidate: predict Cu grade from sensor + operational features
# Show scatter matrix of top features vs target
regression_features = ['avg_fe_grade_pct', 'avg_xrf_confidence', 'surface_volume_correlation',
                       'n_buckets', 'payload_tonnes', 'cycle_time_minutes']

sample_df = df_loads[regression_features + ['avg_cu_grade_pct', 'diversion_type']].dropna().sample(
    min(3000, len(df_loads)), random_state=42
)

fig = px.scatter_matrix(
    sample_df,
    dimensions=regression_features,
    color='diversion_type',
    title='Scatter Matrix — Regression Features vs Cu Grade (color=diversion type)',
    opacity=0.3,
    height=900, width=1000
)
fig.update_traces(diagonal_visible=False, marker=dict(size=3))
fig.show()

In [0]:
# Classification separability analysis — key features for predicting diversion
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Prepare data
ml_features = ['avg_cu_grade_pct', 'avg_fe_grade_pct', 'avg_xrf_confidence', 
               'surface_volume_correlation', 'n_buckets', 'payload_tonnes',
               'cycle_time_minutes', 'estimated_cu_tonnes', 'load_hour', 'day_of_week']

ml_df = df_loads[ml_features + ['diversion_type']].dropna()
le = LabelEncoder()
ml_df['target'] = le.fit_transform(ml_df['diversion_type'])

X = ml_df[ml_features].values
y = ml_df['target'].values

# Quick feature importance with Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.DataFrame({
    'feature': ml_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importances['feature'], importances['importance'], color='steelblue')
ax.set_xlabel('Feature Importance (Gini)')
ax.set_title(f'Random Forest Feature Importance for Diversion Type Classification\n'
             f'(Classes: {dict(zip(le.classes_, le.transform(le.classes_)))})')
for i, (imp, feat) in enumerate(zip(importances['importance'], importances['feature'])):
    ax.text(imp + 0.002, i, f'{imp:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(f"\nQuick RF accuracy (train set): {rf.score(X, y):.3f}")
print(f"Class distribution: {dict(zip(le.classes_, np.bincount(y)))}")

In [0]:
# Regression analysis: predict Cu grade
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score

reg_features = ['avg_fe_grade_pct', 'avg_xrf_confidence', 'surface_volume_correlation',
                'n_buckets', 'payload_tonnes', 'cycle_time_minutes', 'load_hour']

reg_df = df_loads[reg_features + ['avg_cu_grade_pct']].dropna()
X_reg = reg_df[reg_features].values
y_reg = reg_df['avg_cu_grade_pct'].values

gb = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
scores = cross_val_score(gb, X_reg, y_reg, cv=5, scoring='r2')
print(f"Cu Grade Regression — 5-fold CV R²: {scores.mean():.3f} ± {scores.std():.3f}")

# Fit for residual analysis
gb.fit(X_reg, y_reg)
y_pred = gb.predict(X_reg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_reg, y_pred, alpha=0.2, s=10, c='steelblue')
axes[0].plot([y_reg.min(), y_reg.max()], [y_reg.min(), y_reg.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Cu Grade %')
axes[0].set_ylabel('Predicted Cu Grade %')
axes[0].set_title(f'Cu Grade Regression — Actual vs Predicted (R²={scores.mean():.3f})')

# Residual distribution
residuals = y_reg - y_pred
axes[1].hist(residuals, bins=60, color='salmon', edgecolor='white')
axes[1].axvline(0, color='black', linestyle='--')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

## Summary & ML Recommendations

### Key Findings

**Best Classification Candidate: `diversion_type` prediction from `fact_truck_loads`**
- Features: Cu/Fe grades, XRF confidence, surface-volume correlation, payload, cycle time
- Clear class separability visible in feature distributions
- Directly actionable for automated smart truck diversion decisions
- Multi-class problem (Aligned / Ore from Waste / Waste from Ore)

**Best Regression Candidate: `avg_cu_grade_pct` prediction from `fact_truck_loads`**
- Continuous target with good feature correlations
- Fe grade, XRF confidence, and SV correlation are strong predictors
- Economic impact: better grade estimation → better diversion decisions

**Secondary ML Opportunities:**
- **Binary classification**: `is_diverted` (simpler problem, high baseline accuracy)
- **Measurement quality**: predict `measurement_quality_score` from sensor features in `fact_bucket_measurements`
- **Anomaly detection**: identify sensor degradation patterns from `fact_sensor_performance`

### Data Quality Notes
- Check for class imbalance in `diversion_type` (Aligned likely dominant)
- Surface-volume correlation quality varies — consider as a feature *and* a filter
- Geological domain strongly influences classification accuracy — domain-aware models may outperform global models